# Tutorial: Cascadia OBS Ensemble Picking Workflow

This notebook demonstrates the earthquake phase picking workflow using ensemble machine learning models on Cascadia seismic data. We'll process a small subset of stations and time ranges to illustrate the complete workflow.

## Overview

The picking workflow consists of:
1. **Data Retrieval**: Download seismic waveforms from multiple networks
2. **Preprocessing**: Filter, resample, and window the data
3. **Ensemble Prediction**: Apply multiple pre-trained EQTransformer models
4. **Semblance Calculation**: Combine predictions using ELEP ensemble statistics
5. **Phase Detection**: Identify P and S wave arrivals
6. **Output**: Save picks to CSV format

## Key Parameters

- **twin**: 6000 samples (60 seconds at 100 Hz) - window length for ML models
- **step**: 3000 samples (30 seconds) - overlap between windows
- **l_blnd, r_blnd**: 500 samples each - ignore edge effects in predictions

## 1. Setup and Imports

In [2]:
# Standard library imports
import logging
import os
import sys
import datetime
from datetime import timedelta
import time
import gc

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# ObsPy imports
import obspy
from obspy.clients.fdsn import Client
from obspy.core.utcdatetime import UTCDateTime
from obspy import Stream, Trace
from obspy.signal.trigger import trigger_onset

# Seisbench for ML models
import torch
import seisbench.models as sbm

# PNW data store
import project
from project.mseed import WaveformClient

# ELEP for ensemble statistics
from ELEP.elep.ensemble_statistics import ensemble_statistics
from ELEP.elep.ensemble_coherence import ensemble_semblance 
from ELEP.elep.trigger_func import picks_summary_simple

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set device for PyTorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'project'

In [4]:
# Install pnwstore
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", 
                      "git+https://github.com/niyiyu/pnwstore.git"])


  Cloning https://github.com/niyiyu/pnwstore.git to /tmp/pip-req-build-xdm51nn9


  Running command git clone --filter=blob:none --quiet https://github.com/niyiyu/pnwstore.git /tmp/pip-req-build-xdm51nn9


  Resolved https://github.com/niyiyu/pnwstore.git to commit 6e6c9ec1a877a045d554c67f6c7e5b84374a7200
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


0

In [6]:

import project
from project import WaveformClient

ModuleNotFoundError: No module named 'project'

In [8]:
# Alternative 1: Install in editable mode
!git clone https://github.com/niyiyu/pnwstore.git /tmp/pnwstore
!{sys.executable} -m pip install -e /tmp/pnwstore

# Alternative 2: Add the repo directly to Python path
!git clone https://github.com/niyiyu/pnwstore.git /tmp/pnwstore
sys.path.insert(0, '/tmp/pnwstore')
from pnwstore.mseed import WaveformClient

fatal: destination path '/tmp/pnwstore' already exists and is not an empty directory.
Obtaining file:///tmp/pnwstore
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for project (pyproject.toml) ... done
  Created wheel for project: filename=project-0.1.0-0.editable-py3-none-any.whl size=2070 sha256=707925dc744682945fdd139c7aa27b3f194d7137d796a70c6f1e89fada59a78c
  Stored in directory: /tmp/pip-ephem-wheel-cache-snh3zk1h/wheels/ac/34/13/022bc55856ecfc9ddcd37ebfd70c65759f39ef2fee0c0645ea
Successfully built project
  Attempting uninstall: project
    Found existing installation: project 0.1.0
    Uninstalling project-0.1.0:
      Successfully uninstalled project-0.1.0
fatal: destination path '/tmp/pnwstore' already exists and is not an empty directory.


In [9]:

from pnwstore.mseed import WaveformClient

## 2. Define Parameters and Clients

For this tutorial, we'll focus on:
- **2 stations** from different networks
- **1-2 days** of data
- A subset of the full geographic region

In [10]:
# Initialize FDSN and waveform clients
client_inventory = Client('IRIS')
client_waveform = WaveformClient()
client_ncedc = Client('NCEDC')

# Processing parameters
twin = 6000      # Window length in samples (60 sec at 100 Hz)
step = 3000      # Step between windows (30 sec overlap)
l_blnd = 500     # Left blind zone (samples to ignore at window start)
r_blnd = 500     # Right blind zone (samples to ignore at window end)

# Detection thresholds
p_thrd = 0.05    # P-wave detection threshold
s_thrd = 0.05    # S-wave detection threshold

# Filter parameters
freqmin = 4.0    # Hz
freqmax = 15.0   # Hz
target_fs = 100  # Target sampling rate (Hz)

# Tutorial parameters - LIMITED DATA FOR DEMONSTRATION
year = 2011
start_date = datetime.datetime(year, 3, 15)  # March 15, 2011
end_date = datetime.datetime(year, 3, 16)    # Process 1 day

# Geographic bounds (limited region)
minlat, maxlat = 44.0, 46.0
minlon, maxlon = -126.0, -124.0

# Output directory
output_dir = "./tutorial_picks/"
os.makedirs(output_dir, exist_ok=True)

print(f"Tutorial Configuration:")
print(f"  Time range: {start_date} to {end_date}")
print(f"  Geographic bounds: {minlat}-{maxlat}°N, {minlon}-{maxlon}°E")
print(f"  Output directory: {output_dir}")

Tutorial Configuration:
  Time range: 2011-03-15 00:00:00 to 2011-03-16 00:00:00
  Geographic bounds: 44.0-46.0°N, -126.0--124.0°E
  Output directory: ./tutorial_picks/


## 3. Query Station Inventory

Retrieve available stations in our limited geographic region during the specified time period.

In [11]:
# Query station inventory
print("Querying station inventory...")
inventory = client_inventory.get_stations(
    network="UW,UO,7D,C8",  # Limited networks for tutorial
    station="*",
    minlatitude=minlat,
    maxlatitude=maxlat,
    minlongitude=minlon,
    maxlongitude=maxlon,
    starttime=start_date.strftime('%Y%m%d'),
    endtime=end_date.strftime('%Y%m%d'),
    level='station'
)

# Extract station information
stations_info = []
for network in inventory:
    for station in network:
        stations_info.append({
            'network': network.code,
            'station': station.code,
            'latitude': station.latitude,
            'longitude': station.longitude,
            'elevation': station.elevation
        })

stations_df = pd.DataFrame(stations_info)
print(f"\nFound {len(stations_df)} stations:")
print(stations_df.head(10))

# For this tutorial, let's pick just 2 stations for demonstration
if len(stations_df) > 2:
    tutorial_stations = stations_df.iloc[:2]
else:
    tutorial_stations = stations_df

print(f"\nTutorial will process {len(tutorial_stations)} stations")

Querying station inventory...

Found 1 stations:
  network station  latitude  longitude  elevation
0      UW    NEWO  44.62392 -124.04622        4.3

Tutorial will process 1 stations


## 4. Load Pre-trained Models

We use an ensemble of 5 different EQTransformer models trained on different datasets:
- **original (PNW)**: Pacific Northwest data
- **ethz**: Swiss Seismological Service data
- **instance**: Global dataset
- **scedc**: Southern California
- **stead**: STanford EArthquake Dataset (global)

The diversity of training datasets helps improve generalization.

In [13]:
print("Loading pre-trained EQTransformer models...")
pretrain_list = ['original', 'ethz', 'instance', 'scedc', 'stead']

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Load models (they will be downloaded if not cached)
models = {}
for pretrain in pretrain_list:
    print(f"  Loading {pretrain} model...")
    model = sbm.EQTransformer.from_pretrained(pretrain)
    model.to(device)
    model.eval()
    models[pretrain] = model

print("\nAll models loaded successfully!")

Loading pre-trained EQTransformer models...
  Loading original model...
  Loading ethz model...
  Loading instance model...
  Loading scedc model...
  Loading stead model...

All models loaded successfully!


## 5. Define Helper Functions

These functions handle the core processing workflow.

In [14]:
def pred_trigger_pick(pred, source_trace, label, thrd=0.1, **kwargs):
    """
    Detect phase arrivals from model predictions using simple threshold triggering.
    
    Parameters:
    -----------
    pred : np.ndarray
        Model prediction time series (probability values)
    source_trace : obspy.Trace
        Original trace for timing information
    label : str
        Phase label ('P' or 'S')
    thrd : float
        Detection threshold
    
    Returns:
    --------
    picks : pd.DataFrame
        DataFrame containing detected picks
    """
    if len(pred) < 300:
        raise ValueError('Insufficient samples in prediction')
    if not isinstance(source_trace, Trace):
        raise TypeError('source_trace must be obspy.Trace')
    
    # Zero out initial samples to avoid edge effects
    pred[:300] = 0.
    
    # Find trigger windows
    triggers = trigger_onset(pred, thrd, thrd, **kwargs)
    
    # Extract timing information
    t0 = source_trace.stats.starttime
    sr = source_trace.stats.sampling_rate
    trace_id = source_trace.id
    
    picks = []
    for s0, s1 in triggers:
        trigger_onset_time = t0 + s0/sr
        peak_idx = s0 + np.argmax(pred[s0:s1+1])
        pick_time = t0 + peak_idx/sr
        trigger_offset_time = t0 + s1/sr
        peak_value = np.max(pred[s0:s1+1])
        
        # Parse trace ID
        parts = trace_id[:-1].split('.')
        line = parts + [label, t0, trigger_onset_time, pick_time, 
                       trigger_offset_time, peak_value, thrd]
        picks.append(line)
    
    picks_df = pd.DataFrame(
        data=picks,
        columns=['network', 'station', 'location', 'band_inst', 'label',
                'trace_starttime', 'trigger_onset', 'pick_time',
                'trigger_offset', 'max_prob', 'thresh_prob']
    )
    return picks_df


def stacking(data, npts, l_blnd, r_blnd, nseg):
    """
    Stack overlapping window predictions using maximum value selection
    while respecting blind zones.
    
    Parameters:
    -----------
    data : np.ndarray
        Predictions from windowed processing [nseg, twin]
    npts : int
        Total number of points in output
    l_blnd, r_blnd : int
        Left and right blind zone sizes
    nseg : int
        Number of segments
    
    Returns:
    --------
    stack : np.ndarray
        Stacked continuous prediction
    """
    _data = data.copy()
    stack = np.full(npts, np.nan, dtype=np.float32)
    
    # Blind the edge zones
    _data[:, :l_blnd] = np.nan
    _data[:, -r_blnd:] = np.nan
    
    # Initialize with first window
    stack[:twin] = _data[0, :]
    
    # Stack remaining windows using max value
    for iseg in range(nseg - 1):
        idx = step * (iseg + 1)
        stack[idx:idx + twin] = np.nanmax(
            [stack[idx:idx + twin], _data[iseg + 1, :]], 
            axis=0
        )
    
    return stack


print("Helper functions defined successfully!")

Helper functions defined successfully!


## 6. Main Processing Function

This function implements the complete workflow for a single station-day.

In [15]:
def process_station_day(network, station, lat, lon, elev, t1, t2, 
                       output_dir, models, verbose=True):
    """
    Process one station for one day using ensemble ML picking.
    
    Parameters:
    -----------
    network, station : str
        Station identifiers
    lat, lon, elev : float
        Station coordinates
    t1, t2 : datetime or UTCDateTime
        Start and end times
    output_dir : str
        Directory for output CSV files
    models : dict
        Dictionary of loaded ML models
    verbose : bool
        Print progress messages
    
    Returns:
    --------
    picks_df : pd.DataFrame
        DataFrame containing all detected picks
    """
    # Convert to UTCDateTime if needed
    if not isinstance(t1, UTCDateTime):
        t1 = UTCDateTime(t1)
        t2 = UTCDateTime(t2)
    
    if verbose:
        print(f"\nProcessing {network}.{station} on {t1.date}")
    
    # Define output filename
    tstring1 = t1.strftime('%Y%m%d')
    tstring2 = t2.strftime('%Y%m%d')
    save_file = os.path.join(output_dir, f"{network}_{station}_{tstring1}_{tstring2}.csv")
    
    # Skip if already processed
    if os.path.exists(save_file):
        print(f"  File already exists: {save_file}")
        return pd.read_csv(save_file)
    
    # Step 1: Download waveforms
    if verbose:
        print("  Step 1: Downloading waveforms...")
    
    try:
        if network in ['NC', 'BK']:
            sdata = client_ncedc.get_waveforms(
                network=network, station=station, location="*", channel="*",
                starttime=t1, endtime=t2
            )
        else:
            sdata = client_waveform.get_waveforms(
                network=network, station=station, channel="*",
                year=t1.strftime('%Y'), month=t1.strftime('%m'),
                day=t1.strftime('%d')
            )
    except Exception as e:
        print(f"  WARNING: No data available - {e}")
        return None
    
    # Step 2: Select appropriate channels
    if verbose:
        print("  Step 2: Selecting channels...")
    
    # Check for vertical component (required)
    if not sdata.select(channel='??Z'):
        print("  WARNING: No vertical component found")
        return None
    
    # Prioritize high-rate channels: HH > BH > EH
    selected = Stream()
    if sdata.select(channel="HH?"):
        selected = sdata.select(channel="HH?")
        if verbose:
            print("    Using HH channels")
    elif sdata.select(channel="BH?"):
        selected = sdata.select(channel="BH?")
        if verbose:
            print("    Using BH channels")
    elif sdata.select(channel="EH?"):
        selected = sdata.select(channel="EH?")
        if verbose:
            print("    Using EH channels")
    else:
        print("  WARNING: No suitable channels found")
        return None
    
    if len(selected) == 0:
        print("  WARNING: No data in selected channels")
        return None
    
    # Check for constant data
    if np.abs(np.mean(selected[0].data[1:] - selected[0].data[:-1])) <= 1e-8:
        print("  WARNING: Constant/no data in stream")
        return None
    
    # Step 3: Preprocess waveforms
    if verbose:
        print("  Step 3: Preprocessing waveforms...")
    
    # Filter and resample
    selected.detrend('demean')
    selected.detrend('linear')
    selected.filter(type='bandpass', freqmin=freqmin, freqmax=freqmax, zerophase=True)
    selected.resample(target_fs)
    selected.merge(fill_value='interpolate')
    
    # Trim to common time window
    max_starttime = max([tr.stats.starttime for tr in selected])
    min_endtime = min([tr.stats.endtime for tr in selected])
    for tr in selected:
        tr.trim(starttime=max_starttime, endtime=min_endtime, nearest_sample=True)
    
    # Order components: Z, E/1, N/2
    ordered = Stream()
    for comp in ['Z', '[1E]', '[2N]']:
        comp_traces = selected.select(channel=f'??{comp}')
        if len(comp_traces) > 0:
            ordered += comp_traces[0]
    
    if len(ordered) < 3:
        print(f"  WARNING: Only {len(ordered)} components available")
    
    selected = ordered
    
    # Step 4: Window the data
    if verbose:
        print("  Step 4: Creating overlapping windows...")
    
    arr_sdata = np.array([tr.data for tr in selected[:3]])  # Ensure 3 components
    if arr_sdata.shape[0] < 3:
        # Pad with zeros if needed
        arr_sdata = np.vstack([
            arr_sdata,
            np.zeros((3 - arr_sdata.shape[0], arr_sdata.shape[1]))
        ])
    
    npts = arr_sdata.shape[1]
    nseg = int(np.floor((npts - twin) / step)) + 1
    
    # Create windows with tapering
    windows_std = np.zeros((nseg, 3, twin), dtype=np.float32)
    windows_max = np.zeros((nseg, 3, twin), dtype=np.float32)
    tap = 0.5 * (1 + np.cos(np.linspace(np.pi, 2 * np.pi, 6)))
    
    for iseg in range(nseg):
        idx = iseg * step
        window = arr_sdata[:, idx:idx + twin]
        window = window - np.mean(window, axis=-1, keepdims=True)
        
        # Standard deviation normalization
        windows_std[iseg] = window / (np.std(window) + 1e-10)
        
        # Max normalization
        windows_max[iseg] = window / (np.max(np.abs(window), axis=-1, keepdims=True) + 1e-10)
        
        # Apply taper
        windows_std[iseg, :, :6] *= tap
        windows_std[iseg, :, -6:] *= tap[::-1]
        windows_max[iseg, :, :6] *= tap
        windows_max[iseg, :, -6:] *= tap[::-1]
    
    if verbose:
        print(f"    Created {nseg} windows of shape {windows_std.shape}")
    
    # Step 5: Run ensemble predictions
    if verbose:
        print("  Step 5: Running ensemble predictions...")
    
    batch_pred = np.zeros([2, len(pretrain_list), nseg, twin], dtype=np.float32)
    
    for ipre, pretrain in enumerate(pretrain_list):
        if verbose:
            print(f"    Predicting with {pretrain} model...")
        
        model = models[pretrain]
        
        # Use std normalization for 'original' model, max for others
        if pretrain == 'original':
            windows_tensor = torch.Tensor(windows_std).to(device)
        else:
            windows_tensor = torch.Tensor(windows_max).to(device)
        
        with torch.no_grad():
            predictions = model(windows_tensor)
        
        # predictions: [detection, P, S]
        batch_pred[0, ipre, :] = predictions[1].detach().cpu().numpy()  # P
        batch_pred[1, ipre, :] = predictions[2].detach().cpu().numpy()  # S
    
    # Clean up memory
    del windows_tensor, predictions
    torch.cuda.empty_cache()
    gc.collect()
    
    # Step 6: Calculate ensemble semblance
    if verbose:
        print("  Step 6: Calculating ensemble semblance...")
    
    dt = 1.0 / target_fs
    paras_semblance = {
        'dt': dt,
        'semblance_order': 2,
        'window_flag': True,
        'semblance_win': 0.5,
        'weight_flag': 'max'
    }
    
    smb_pred = np.zeros([2, nseg, twin], dtype=np.float32)
    
    for iseg in range(nseg):
        # P-wave semblance
        smb_pred[0, iseg, :] = ensemble_semblance(
            batch_pred[0, :, iseg, :], paras_semblance
        )
        
        # S-wave semblance
        smb_pred[1, iseg, :] = ensemble_semblance(
            batch_pred[1, :, iseg, :], paras_semblance
        )
    
    # Step 7: Stack predictions
    if verbose:
        print("  Step 7: Stacking windowed predictions...")
    
    smb_p = stacking(smb_pred[0, :], npts, l_blnd, r_blnd, nseg)
    smb_s = stacking(smb_pred[1, :], npts, l_blnd, r_blnd, nseg)
    
    del smb_pred, batch_pred
    
    # Step 8: Detect picks
    if verbose:
        print("  Step 8: Detecting phase arrivals...")
    
    picks_p = pred_trigger_pick(smb_p, selected[0], 'P', thrd=p_thrd)
    picks_s = pred_trigger_pick(smb_s, selected[0], 'S', thrd=s_thrd)
    
    picks_df = pd.concat([picks_p, picks_s], axis=0, ignore_index=True)
    
    if verbose:
        print(f"    Detected {len(picks_p)} P-picks and {len(picks_s)} S-picks")
    
    # Save to CSV
    picks_df.to_csv(save_file, index=False)
    if verbose:
        print(f"  Saved picks to: {save_file}")
    
    return picks_df, smb_p, smb_s, selected


print("Main processing function defined successfully!")

Main processing function defined successfully!


## 7. Process Tutorial Stations

Now let's run the processing on our small subset of stations.

In [16]:
# Process each station
all_picks = []
processing_results = {}

for idx, row in tutorial_stations.iterrows():
    print("=" * 70)
    
    try:
        result = process_station_day(
            network=row['network'],
            station=row['station'],
            lat=row['latitude'],
            lon=row['longitude'],
            elev=row['elevation'],
            t1=start_date,
            t2=end_date,
            output_dir=output_dir,
            models=models,
            verbose=True
        )
        
        if result is not None:
            if len(result) == 4:
                picks_df, smb_p, smb_s, stream = result
                processing_results[f"{row['network']}.{row['station']}"] = {
                    'picks': picks_df,
                    'smb_p': smb_p,
                    'smb_s': smb_s,
                    'stream': stream
                }
                all_picks.append(picks_df)
            else:
                # Already processed, just picks returned
                all_picks.append(result)
    
    except Exception as e:
        print(f"ERROR processing {row['network']}.{row['station']}: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "=" * 70)
print("Processing complete!")
print("=" * 70)


Processing UW.NEWO on 2011-03-15
  Step 1: Downloading waveforms...

Processing complete!


## 8. Analyze Results

Let's examine the picks we detected.

In [ ]:
# Combine all picks
if all_picks:
    combined_picks = pd.concat(all_picks, ignore_index=True)
    
    print(f"Total picks detected: {len(combined_picks)}")
    print(f"  P-wave picks: {len(combined_picks[combined_picks['label'] == 'P'])}")
    print(f"  S-wave picks: {len(combined_picks[combined_picks['label'] == 'S'])}")
    print(f"\nPicks by station:")
    print(combined_picks.groupby(['network', 'station', 'label']).size())
    
    print("\nSample picks:")
    print(combined_picks.head(10))
    
    # Probability distribution
    print("\nProbability statistics:")
    print(combined_picks[['label', 'max_prob']].groupby('label').describe())
else:
    print("No picks detected in this tutorial run.")

## 9. Visualize Results

Plot waveforms with detected picks and ensemble predictions.

In [ ]:
# Visualize results for the first successfully processed station
if processing_results:
    station_key = list(processing_results.keys())[0]
    result = processing_results[station_key]
    
    picks = result['picks']
    smb_p = result['smb_p']
    smb_s = result['smb_s']
    stream = result['stream']
    
    # Create figure
    fig, axes = plt.subplots(5, 1, figsize=(15, 12), sharex=True)
    
    # Time axis (in minutes)
    t0 = stream[0].stats.starttime
    dt = 1.0 / stream[0].stats.sampling_rate
    npts = len(stream[0].data)
    time_min = np.arange(npts) * dt / 60.0  # Convert to minutes
    
    # Plot waveforms (3 components)
    components = ['Z', 'N/E', 'E/N']
    for i in range(min(3, len(stream))):
        axes[i].plot(time_min, stream[i].data, 'k', linewidth=0.5)
        axes[i].set_ylabel(f'{stream[i].stats.channel}\nAmplitude')
        axes[i].grid(True, alpha=0.3)
        
        # Add pick markers
        for _, pick in picks.iterrows():
            pick_time_min = (UTCDateTime(pick['pick_time']) - t0) / 60.0
            color = 'blue' if pick['label'] == 'P' else 'red'
            axes[i].axvline(pick_time_min, color=color, alpha=0.5, 
                          linewidth=2, linestyle='--')
    
    # Plot P-wave ensemble predictions
    time_pred = np.arange(len(smb_p)) * dt / 60.0
    axes[3].plot(time_pred, smb_p, 'b', linewidth=1.5, label='P-wave')
    axes[3].axhline(p_thrd, color='b', linestyle=':', alpha=0.5, label=f'Threshold ({p_thrd})')
    axes[3].set_ylabel('P-wave\nProbability')
    axes[3].grid(True, alpha=0.3)
    axes[3].legend(loc='upper right')
    axes[3].set_ylim([0, 1])
    
    # Plot S-wave ensemble predictions
    axes[4].plot(time_pred, smb_s, 'r', linewidth=1.5, label='S-wave')
    axes[4].axhline(s_thrd, color='r', linestyle=':', alpha=0.5, label=f'Threshold ({s_thrd})')
    axes[4].set_ylabel('S-wave\nProbability')
    axes[4].set_xlabel('Time (minutes from start)')
    axes[4].grid(True, alpha=0.3)
    axes[4].legend(loc='upper right')
    axes[4].set_ylim([0, 1])
    
    # Title
    fig.suptitle(f'{station_key} - {t0.date}\n'
                f'{len(picks[picks["label"]=="P"])} P-picks, '
                f'{len(picks[picks["label"]=="S"])} S-picks',
                fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Save figure
    fig_path = os.path.join(output_dir, f"{station_key}_{start_date.strftime('%Y%m%d')}.png")
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"\nFigure saved to: {fig_path}")

else:
    print("No processed data available for visualization.")

## 10. Plot Pick Time Distribution

Visualize when picks occur throughout the day.

In [ ]:
if all_picks and len(combined_picks) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    
    # Extract pick times
    p_picks = combined_picks[combined_picks['label'] == 'P'].copy()
    s_picks = combined_picks[combined_picks['label'] == 'S'].copy()
    
    if len(p_picks) > 0:
        p_picks['pick_datetime'] = pd.to_datetime(p_picks['pick_time'])
        axes[0].hist(p_picks['pick_datetime'], bins=50, color='blue', alpha=0.7)
        axes[0].set_ylabel('P-wave Pick Count')
        axes[0].grid(True, alpha=0.3)
        axes[0].set_title(f'P-wave Picks Distribution (n={len(p_picks)})')
    
    if len(s_picks) > 0:
        s_picks['pick_datetime'] = pd.to_datetime(s_picks['pick_time'])
        axes[1].hist(s_picks['pick_datetime'], bins=50, color='red', alpha=0.7)
        axes[1].set_ylabel('S-wave Pick Count')
        axes[1].set_xlabel('Time')
        axes[1].grid(True, alpha=0.3)
        axes[1].set_title(f'S-wave Picks Distribution (n={len(s_picks)})')
    
    plt.tight_layout()
    plt.show()
    
    # Probability distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    if len(p_picks) > 0:
        axes[0].hist(p_picks['max_prob'], bins=30, color='blue', alpha=0.7, edgecolor='black')
        axes[0].axvline(p_picks['max_prob'].median(), color='darkblue', 
                       linestyle='--', linewidth=2, label=f'Median: {p_picks["max_prob"].median():.3f}')
        axes[0].set_xlabel('Probability')
        axes[0].set_ylabel('Count')
        axes[0].set_title('P-wave Pick Probabilities')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
    
    if len(s_picks) > 0:
        axes[1].hist(s_picks['max_prob'], bins=30, color='red', alpha=0.7, edgecolor='black')
        axes[1].axvline(s_picks['max_prob'].median(), color='darkred', 
                       linestyle='--', linewidth=2, label=f'Median: {s_picks["max_prob"].median():.3f}')
        axes[1].set_xlabel('Probability')
        axes[1].set_ylabel('Count')
        axes[1].set_title('S-wave Pick Probabilities')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

else:
    print("No picks available for distribution plots.")

## Summary and Next Steps

### What We've Demonstrated:

1. **Data Retrieval**: Downloaded waveforms from IRIS/PNWStore for specific stations
2. **Preprocessing**: Filtered, resampled, and windowed the data appropriately
3. **Ensemble ML**: Applied 5 different EQTransformer models and combined predictions
4. **Semblance**: Used ensemble semblance to enhance signal coherence
5. **Detection**: Identified P and S wave arrivals above threshold
6. **Visualization**: Plotted waveforms with picks and probability functions

### Scaling to Production:

The full production scripts (`parallel_pick_*.py`) extend this workflow by:

1. **Parallel Processing**: Using Dask to process multiple station-days simultaneously
2. **Full Networks**: Processing all available stations in the region
3. **Full Time Range**: Processing entire years of data
4. **Error Handling**: Robust exception handling for data gaps and failures
5. **Channel Prioritization**: Different scripts handle different channel types (HH/BH vs EH)
6. **Resource Management**: Efficient memory cleanup and GPU utilization

### Key Files:

- `picking_utils.py`: Core functions for standard channels (HH/BH)
- `picking_utils_prio_EH.py`: Specialized for EH (analog) channels
- `parallel_pick_20XX_*.py`: Year-specific processing scripts

### Typical Usage Pattern:

```bash
# Process one year with standard channels
python parallel_pick_2011_HH_BH.py

# Process specific station ranges with EH priority
python parallel_pick_2011_123_127_EH.py
```

### Adjustable Parameters:

- **Detection thresholds** (`p_thrd`, `s_thrd`): Lower values = more picks but more false positives
- **Window parameters** (`twin`, `step`): Affect computational cost and temporal resolution
- **Filter bands** (`freqmin`, `freqmax`): Tune for different seismic frequencies
- **Number of workers**: Balance between parallelism and memory usage